# Week 1 – Exercise 1 – Individual – Metadata Creation
## Name: Anushka Sawant
## Matriculation Number: 100006644

# Week 1 — Exercise 1: Individual Metadata Creation

## Goal
- Practice creating metadata by applying what you learned in class to a dataset of your choice.

## Task
- Select any publicly available dataset, for example from Kaggle, Zenodo, or Figshare, and create the following:
- DataCite metadata in XML format
- schema.org metadata in JSON-LD format using @type: Dataset


## Submission / GitHub Workflow

Upload your individual exercise to your own GitHub repository
Organize your repository clearly by week and exercise
For group projects, all group members may collaborate and keep their work in GitHub
Only one group representative should submit the final GitHub repository link
The README file must clearly include the names of all group members

## File Naming
Please use clear filenames, for example:

- yourdataset_datacite.xml
- yourdataset_schemaorg.json

## Dataset used: Credit Risk Benchmark Dataset (Kaggle)
https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset

In [1]:
from pathlib import Path
import pandas as pd
import hashlib
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from pathlib import Path
from urllib.request import urlretrieve
from lxml import etree
from google.colab import files

## 1. Load Dataset

In [2]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AnushkaSDBI/Data-Management/main"

required_files = [
    ("Data/Credit%20Risk%20Benchmark%20Dataset.csv", "Data/Credit%20Risk%20Benchmark%20Dataset.csv"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Bootstrap complete.


In [3]:
repo_root = Path().resolve()
csv_path = repo_root / "Data" / "Credit%20Risk%20Benchmark%20Dataset.csv"

print("Expected CSV path:", csv_path)
print("File exists:", csv_path.exists())

df = pd.read_csv(csv_path)
display(df)

Expected CSV path: /content/Data/Credit%20Risk%20Benchmark%20Dataset.csv
File exists: True


,rev_util,age,late_30_59,debt_ratio,monthly_inc,open_credit,late_90,real_estate,late_60_89,dependents,dlq_2yrs
0,0.006999,38.0,0.0,0.302150,5440.0,4.0,0.0,1.0,0.0,3.0,0
1,0.704592,63.0,0.0,0.471441,8000.0,9.0,0.0,1.0,0.0,0.0,0
2,0.063113,57.0,0.0,0.068586,5000.0,17.0,0.0,0.0,0.0,0.0,0
3,0.368397,68.0,0.0,0.296273,6250.0,16.0,0.0,2.0,0.0,0.0,0
4,1.000000,34.0,1.0,0.000000,3500.0,0.0,0.0,0.0,0.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...
16709,1.000000,46.0,0.0,170.398010,401.0,3.0,2.0,0.0,0.0,2.0,1
16710,1.135552,41.0,2.0,0.845887,7500.0,12.0,0.0,4.0,1.0,0.0,1
16711,0.920107,31.0,1.0,0.176732,1125.0,4.0,1.0,0.0,0.0,0.0,1
16712,0.983825,55.0,0.0,0.064116,4600.0,2.0,1.0,0.0,0.0,6.0,1


## 2. Dataset Profiling

In [4]:
file_size = csv_path.stat().st_size

sha = hashlib.sha256()
with csv_path.open('rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        sha.update(chunk)

checksum = sha.hexdigest()

summary = {
    'rows': len(df),
    'columns': df.shape[1],
    'size_bytes': file_size,
    'sha256': checksum
}

print(json.dumps(summary, indent=2))

{
  "rows": 16714,
  "columns": 11,
  "size_bytes": 1024867,
  "sha256": "021521667577dd613c048c5e5c3e754e38b34842e5b91661b93df7e633ede578"
}


## 3. Column Schema Inspection

In [5]:
for c in df.columns:
    print(c, '->', df[c].dtype)

rev_util -> float64
age -> float64
late_30_59 -> float64
debt_ratio -> float64
monthly_inc -> float64
open_credit -> float64
late_90 -> float64
real_estate -> float64
late_60_89 -> float64
dependents -> float64
dlq_2yrs -> int64


## 4. Metadata Dictionary

In [6]:
META = {
    'identifier': '10.1234/credit-risk-benchmark',
    'title': 'Credit Risk Benchmark Dataset',
    'publisher': 'Kaggle',
    'publicationYear': '2023',
    'creators': [{'name': 'adilshamim8'}],
    'description': f'Dataset with {len(df)} rows and {df.shape[1]} columns for credit risk prediction.',
    'subjects': list(df.columns),
    'format': 'text/csv',
    'rows': len(df),
    'columns': df.shape[1],
    'size_bytes': file_size,
    'sha256': checksum
}

## 5. DataCite XML Generation

In [7]:
NS = 'http://datacite.org/schema/kernel-4'
resource = etree.Element(f'{{{NS}}}resource')

etree.SubElement(resource, 'identifier', identifierType='DOI').text = META['identifier']

creators = etree.SubElement(resource, 'creators')
for c in META['creators']:
    creator = etree.SubElement(creators, 'creator')
    etree.SubElement(creator, 'creatorName').text = c['name']

titles = etree.SubElement(resource, 'titles')
etree.SubElement(titles, 'title').text = META['title']

etree.SubElement(resource, 'publisher').text = META['publisher']
etree.SubElement(resource, 'publicationYear').text = META['publicationYear']

subjects = etree.SubElement(resource, 'subjects')
for s in META['subjects']:
    etree.SubElement(subjects, 'subject').text = str(s)

desc = etree.SubElement(resource, 'descriptions')
etree.SubElement(desc, 'description', descriptionType='Abstract').text = META['description']

xml_bytes = etree.tostring(resource, pretty_print=True, xml_declaration=True, encoding='UTF-8')
open('credit_risk_datacite.xml','wb').write(xml_bytes)

print(xml_bytes.decode())

<?xml version='1.0' encoding='UTF-8'?>
<ns0:resource xmlns:ns0="http://datacite.org/schema/kernel-4">
  <identifier identifierType="DOI">10.1234/credit-risk-benchmark</identifier>
  <creators>
    <creator>
      <creatorName>adilshamim8</creatorName>
    </creator>
  </creators>
  <titles>
    <title>Credit Risk Benchmark Dataset</title>
  </titles>
  <publisher>Kaggle</publisher>
  <publicationYear>2023</publicationYear>
  <subjects>
    <subject>rev_util</subject>
    <subject>age</subject>
    <subject>late_30_59</subject>
    <subject>debt_ratio</subject>
    <subject>monthly_inc</subject>
    <subject>open_credit</subject>
    <subject>late_90</subject>
    <subject>real_estate</subject>
    <subject>late_60_89</subject>
    <subject>dependents</subject>
    <subject>dlq_2yrs</subject>
  </subjects>
  <descriptions>
    <description descriptionType="Abstract">Dataset with 16714 rows and 11 columns for credit risk prediction.</description>
  </descriptions>
</ns0:resource>



## 6. schema.org JSON-LD

In [8]:
schema = {
    '@context': 'https://schema.org/',
    '@type': 'Dataset',
    'name': META['title'],
    'description': META['description'],
    'creator': [{'@type': 'Person', 'name': c['name']} for c in META['creators']],
    'publisher': {'@type': 'Organization', 'name': META['publisher']},
    'keywords': META['subjects'],
    'encodingFormat': META['format'],
    'url': 'https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset'
}

json.dump(schema, open('credit_risk_schemaorg.json','w'), indent=2)

print(json.dumps(schema, indent=2))

{
  "@context": "https://schema.org/",
  "@type": "Dataset",
  "name": "Credit Risk Benchmark Dataset",
  "description": "Dataset with 16714 rows and 11 columns for credit risk prediction.",
  "creator": [
    {
      "@type": "Person",
      "name": "adilshamim8"
    }
  ],
  "publisher": {
    "@type": "Organization",
    "name": "Kaggle"
  },
  "keywords": [
    "rev_util",
    "age",
    "late_30_59",
    "debt_ratio",
    "monthly_inc",
    "open_credit",
    "late_90",
    "real_estate",
    "late_60_89",
    "dependents",
    "dlq_2yrs"
  ],
  "encodingFormat": "text/csv",
  "url": "https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset"
}


## 7. Validation
Checking XML and JSON-LD structure

In [9]:
xml_check = etree.fromstring(open('credit_risk_datacite.xml','rb').read())
json_check = json.load(open('credit_risk_schemaorg.json'))

print('XML root:', xml_check.tag)
print('JSON type:', json_check['@type'])

XML root: {http://datacite.org/schema/kernel-4}resource
JSON type: Dataset


In [12]:
# download XML
files.download("credit_risk_datacite.xml")

# download JSON-LD
files.download("credit_risk_schemaorg.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This notebook demonstrates a full metadata pipeline including dataset profiling, DataCite XML generation, schema.org JSON-LD creation, and validation on 2024 Credit Risk Dataset.